#request resources + use analy environment


salloc --partition test --time 0-04:00 --mem 20gb
module load python
mamba activate analy
cd /n/home07/than157/desktop/done-large_projects/learn-better/evolm/finetune/llama-factory/
jupyter notebook --no-browser --ip=0.0.0.0 --port=8888

In [1]:
import json
import pandas as pd
from pathlib import Path
import re
from tqdm import tqdm


# download dataset

In [2]:
# #make directory
# DATA_DIR="/n/netscratch/doshi-velez_lab/Lab/th/race_dataset"
# !mkdir -p $DATA_DIR 
# #download dataset
# !wget http://www.cs.cmu.edu/~glai1/data/race/RACE.tar.gz -P $DATA_DIR
# #unzip dataset 
# !tar -xzf $DATA_DIR/RACE.tar.gz -C $DATA_DIR 
# #remove the zipped file
# !rm $DATA_DIR/RACE.tar.gz 

# load dataset as df

In [3]:
def load_txt_files_in_folder(folder_path):
    '''
    Given a folder with text files, load all text files as one df
    '''

    #get all txt files in folder
    txt_files = Path(folder_path).glob("*.txt")

    df_lst = []

    for file in txt_files:
        #load text file as json file
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f) #dict with keys: dict_keys(['answers', 'options', 'questions', 'article', 'id'])

        #convert to df
        df_file = pd.DataFrame({
            "question": data["questions"],
            "options": data["options"],
            "answer": data["answers"]
        })

        df_file['article'] = data["article"]
        df_file['id'] = data["id"]

        #append to list
        df_lst.append(df_file)

    #concatenate all dfs
    df = pd.concat(df_lst)

    return df

### training set -- combine training + val set

In [4]:
#load training set -- high school + middle school
folder_path = '/n/netscratch/doshi-velez_lab/Lab/th/race_dataset/RACE/train/high'
train_high_school = load_txt_files_in_folder(folder_path)

folder_path = '/n/netscratch/doshi-velez_lab/Lab/th/race_dataset/RACE/train/middle'
train_middle_school = load_txt_files_in_folder(folder_path)

folder_path = '/n/netscratch/doshi-velez_lab/Lab/th/race_dataset/RACE/dev/high'
val_high_school = load_txt_files_in_folder(folder_path)

folder_path = '/n/netscratch/doshi-velez_lab/Lab/th/race_dataset/RACE/dev/middle'
val_middle_school = load_txt_files_in_folder(folder_path)

In [5]:
#create full training set
train_high_school['school_level'] = 'high'
train_middle_school['school_level'] = 'middle'
val_high_school['school_level'] = 'high'
val_middle_school['school_level'] = 'middle'

train_df = pd.concat([train_high_school, train_middle_school, val_high_school, val_middle_school])

print('train_high_school', train_high_school.shape)
print('train_middle_school', train_middle_school.shape)
print('val_high_school', val_high_school.shape)
print('val_middle_school', val_middle_school.shape)
print('train_df', train_df.shape)


train_high_school (62445, 6)
train_middle_school (25421, 6)
val_high_school (3451, 6)
val_middle_school (1436, 6)
train_df (92753, 6)


### test set

In [6]:
#load training set -- high school + middle school
folder_path = '/n/netscratch/doshi-velez_lab/Lab/th/race_dataset/RACE/test/high'
test_high_school = load_txt_files_in_folder(folder_path)

folder_path = '/n/netscratch/doshi-velez_lab/Lab/th/race_dataset/RACE/test/middle'
test_middle_school = load_txt_files_in_folder(folder_path)

In [7]:
#create full test set
test_high_school['school_level'] = 'high'
test_middle_school['school_level'] = 'middle'

test_df = pd.concat([test_high_school, test_middle_school])

print('test_high_school', test_high_school.shape)
print('test_middle_school', test_middle_school.shape)
print('test_df', test_df.shape)

test_high_school (3498, 6)
test_middle_school (1436, 6)
test_df (4934, 6)


# process train set

### clean up dataset

In [8]:
train_df.head()

,question,options,answer,article,id,school_level
0,The husband likes shopping because _ .,"[he has much money., he likes the shops., he l...",C,My husband is a born shopper. He loves to look...,high1.txt,high
1,They never go shopping together because _ .,"[their ways of shopping are quite different, t...",A,My husband is a born shopper. He loves to look...,high1.txt,high
2,Jimmy can't do the shopping well because _ .,"[he is young, he is absent-minded, he often lo...",B,My husband is a born shopper. He loves to look...,high1.txt,high
3,Jimmy didn't buy what his mother wanted becaus...,"[the shop was closed that day, the policeman s...",C,My husband is a born shopper. He loves to look...,high1.txt,high
0,Which of the following is true of the introduc...,"[The Britons got expensive tea from India., Te...",B,Tea drinking was common in China for nearly on...,high10.txt,high


In [9]:
#look at all the different endings + counts for each ending
train_endings_counts = pd.DataFrame(train_df['question'].str.strip().str[-1].value_counts())
train_endings_counts

,count
question,
.,46542
?,42676
_,2403
e,151
t,142
"""",140
s,103
o,67
],47


In [10]:
#visually inspect each question ending
ending = '['

filtered_df = train_df[
    train_df["question"].str.strip().str.endswith((ending))
]

print(filtered_df.shape)

for idx in range(filtered_df.shape[0]):
    print(idx)
    print(filtered_df.iloc[idx]['question'])
    print(filtered_df.iloc[idx]['options'])
    # print(filtered_df.iloc[idx]['article'])
    print('-----')

(6, 6)
0
The passage is written to   _  .[
['talk about touring experiences', 'attract people to the tour', 'talk about the history of Africa', 'introduce places of interest in Africa']
-----
1
According to Ben Rhodes, which is the purpose of Barack Obama's visit to Cuba?[
['To bring negative change for Cubans.', "To carry out his vision for Cuba's future.", 'To break silence and no communications.', 'To give Mr Castro a list of political views.']
-----
2
One who gets the job will teach   _  .[
['French', 'English', 'Chinese', 'German']
-----
3
What is happening to the wallet?[
['It is disappearing.', 'It is becoming costly.', 'It is being fattened.', 'It is changing in style.']
-----
4
All the following statements are the most special natures of Death Valley except_.[
['it has the hottest ground in the world', 'it has the largest number of desert plants', 'it is the driest place in North America', 'it is the lowest point in the western hemisphere']
-----
5
What is Lily doing at the bi

In [11]:
#clean up question: remove wrong strings at the end of the question

train_df['question_clean'] = train_df['question']

#if ending is in list, remove this string from the question
endings_to_remove_train = ['(   )', '*', 'ks5u', 'k*s5*u', 'www.ks5u.com', 'www.szzx100.com', 
'------', '----', '--', '-', 's6t----', "'", '[:',
'http://www.ks5u.com/gaokao/beijing/', '/', '[]', '[',
'zxxk', ';', '>', 'WWW.K**S*858$$U.COM', '^', 'K^S*5U.C', 
'  www.k@s@5@u.com_####', '0', 'EUR', '`', '|',
]

for ending in endings_to_remove_train:
    pattern = re.escape(ending) + r'\s*$'
    train_df['question_clean'] = train_df['question_clean'].str.replace(
        pattern,
        '',
        regex=True
    )

In [12]:
#clean up question: replace wrong strings with correct strings
replace_dict = {
    'rights7': 'rights?',
    'TRUE7': 'TRUE?',
    'governments7': 'governments?',
    'This passage is written mainly for  _  .w': 'This passage is written mainly for  _  .',
    'throats"?A': 'throats?',
    'The writer suggests that Ha nk Viscardi': 'The writer suggests that Hank Viscardi'
}

for wrong_str, correct_str in replace_dict.items(): 
    train_df['question_clean'] = train_df['question_clean'].str.replace(wrong_str, correct_str)

In [13]:
#remove this question -- not a question
not_question = "But when you see each other, you can share something you didn't have before he or she left: You can introduce him or her to your new friends!"
train_df = train_df[train_df["question"] != not_question]

In [14]:
#clean up options

#remove bad strings from the end of options
def clean_options(row, endings_to_remove):
    options = row['options']

    options_clean = []

    for o in options:
        #if options ends with a string in endings_to_remove, remove that string ending
        for ending in endings_to_remove:
            if o.endswith(ending):
                    o = o[:-len(ending)]
        
        options_clean.append(o)

    return options_clean

In [15]:
#clean up options: emove bad strings from the end of options
train_df['options_clean'] = train_df.apply(clean_options, axis=1, endings_to_remove=endings_to_remove_train)

In [16]:
#check
train_endings_counts_clean = pd.DataFrame(train_df['question_clean'].str.strip().str[-1].value_counts())
train_endings_counts_clean

,count
question_clean,
.,46561
?,42719
_,2454
e,155
t,150
"""",140
s,107
o,68
],45


### write sft_input and sft_output

In [17]:
#all questions have 4 choices
train_df['n_choices'] = train_df.apply(lambda row: len(row['options']), axis=1).tolist()
print(set(train_df['n_choices']))

{4}


In [18]:
letters = ['A', 'B', 'C', 'D']

def write_sft_input(row):
    #write string for options
    options = row['options_clean']
    
    options_string = ""
    for i, option in enumerate(options):
        options_string += f"\n{letters[i]}. {option}"

    #write sft input
    passage = row['article']
    question = row['question_clean']

    sft_input = f'Answer the question below based only on the provided passage.\n\nPassage: {passage}\n\nQuestion: {question}{options_string}'

    return sft_input

In [19]:
letter_to_number = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

def write_sft_output(row):
    correct_answer = row['answer']
    idx_correct_answer = letter_to_number[correct_answer]
    text_of_correct_answer = row['options_clean'][idx_correct_answer]

    if text_of_correct_answer.endswith('.'):
        period_or_not = ''
    else:
        period_or_not = '.'

    sft_output = f'{text_of_correct_answer}{period_or_not} Therefore, the answer is {correct_answer}.'
    return sft_output

In [20]:
train_df['sft_input'] = train_df.apply(write_sft_input, axis=1)
train_df['sft_output'] = train_df.apply(write_sft_output, axis=1)


### check length of sft_input+sft_output

In [21]:
train_df['total_length'] = train_df['sft_input'].str.len() + train_df['sft_output'].str.len()
print(train_df['total_length'].describe())

#keep only rows where total length is less than max length
max_length = 2048*3
train_df = train_df[train_df['total_length'] < max_length].reset_index(drop=True)
print(train_df['total_length'].describe())


count    92752.000000
mean      1901.063923
std        617.230375
min        209.000000
25%       1517.000000
50%       1939.000000
75%       2254.000000
max       6847.000000
Name: total_length, dtype: float64
count    92737.000000
mean      1900.305930
std        614.392098
min        209.000000
25%       1517.000000
50%       1939.000000
75%       2254.000000
max       6112.000000
Name: total_length, dtype: float64


### save as json format for SFT

In [22]:
#look at example
idx = 745
print('sft_input:')
print(train_df.iloc[idx]['sft_input'])

print('\n\nsft_output:')
print(train_df.iloc[idx]['sft_output'])

sft_input:
Answer the question below based only on the provided passage.

Passage: When the dog named Judy spotted the first sheep in her life, she did what comes naturally. The four-year-old dog set off racing after the sheep across several fields and, being a city animal, lost both her sheep and her sense of direction. Then she ran along the edge of cliff( ) and fell 100 feet, bouncing off a rock into the sea.
Her owner Mike Holden panicked and celled the coastguard of Cornwall, who turned up in seconds . Six volunteers slid down the cliff with the help of a rope but gave up all hope of finding her alive after a 90-minute search.
Three days later, a hurricane hit the coast near Cornwall. Mr. Holden returned home from his holiday upset and convinced his pet was dead. He comforted himself with the thought she had died in the most beautiful part of the country.
For the next two weeks, the Holdens were heartbroken . Then, one day, the phone rang and Steve Tregear, the coastguard of Cornw

In [23]:
### create json file

#format data for sft
data = []

for idx in tqdm(range(train_df.shape[0])):
    row = train_df.iloc[idx]
    item = {
        "instruction": row["sft_input"],
        "input": "",
        "output": row["sft_output"]
    }
    data.append(item)

#save to JSON file
with open("data/race.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Final # of samples in json file:", len(data))
print("Complete!")

 27%|███████████████▏                                        | 25152/92737 [00:00<00:02, 31272.51it/s]

100%|████████████████████████████████████████████████████████| 92737/92737 [00:02<00:00, 31264.46it/s]


Final # of samples in json file: 92737
Complete!


# process test set

### clean up dataset

In [24]:
#look at all the different endings + counts for each ending
test_endings_counts = pd.DataFrame(test_df['question'].str.strip().str[-1].value_counts())
test_endings_counts

,count
question,
.,2516
?,2204
_,146
e,14
"""",10
t,8
s,5
-,4
",",3


In [25]:
#visually inspect each question ending
ending = '-'

filtered_df = test_df[
    test_df["question"].str.strip().str.endswith((ending))
]

print(filtered_df.shape)

for idx in range(filtered_df.shape[0]):
    print(idx)
    print(filtered_df.iloc[idx]['question'])
    print(filtered_df.iloc[idx]['options'])
    # print(filtered_df.iloc[idx]['article'])
    print('-----')

(4, 6)
0
This passage was written mainly for   _  .s6t----
['students who study at Milford Central School, New Yorks6t----', 'people who want to attend Earth Day events in the Catskill regions6t----', 'adults who protect the environment in New Yorks6t----', 'people who celebrate Earth Day all over the worlds6t----']
-----
1
If a couple with a five-year-old child go to Frost Valley, they should pay   _  .s6t----
['15 dollars', '20 dollars', '30 dollars', '35 dollarss6t----']
-----
2
If you want to get tickets to the fashion show, you should call   _  .s6t----
['(845) 985-2291', '(518) 829-7516', '(607) 286-7721s6t----', '(607) 547-4488']
-----
3
Where will you go if you only have time on Sunday?s6t----
['Frost Valley.s6t----', 'Milford Central School.', 'Grafton Lakes State Park.', 'Schoharie Crossing State Historic Site.s6t----']
-----


In [26]:
#clean up question: remove wrong strings at the end of the question

test_df['question_clean'] = test_df['question']

#if ending is in list, remove this string from the question
endings_to_remove_test = [
    's6t----', '(, )', '(Line 1, Para. 9)', 'k*s@5%u', '[', '[:Zxxk.Com]',
]

#remove bad strings from the end of question
for ending in endings_to_remove_test:
    pattern = re.escape(ending) + r'\s*$'
    test_df['question_clean'] = test_df['question_clean'].str.replace(
        pattern,
        '',
        regex=True
    )

In [27]:
#clean up options: remove bad strings from the end of options
test_df['options_clean'] = test_df.apply(clean_options, axis=1, endings_to_remove=endings_to_remove_test)

In [28]:
#look at example options -- should now be cleaned
specific_q = 'This passage was written mainly for   _  .'

print('before cleaning:')
print(test_df[test_df['question_clean'] == specific_q]['options'].values)

print('after cleaning:')
print(test_df[test_df['question_clean'] == specific_q]['options_clean'].values)

before cleaning:
[list(['students who study at Milford Central School, New Yorks6t----', 'people who want to attend Earth Day events in the Catskill regions6t----', 'adults who protect the environment in New Yorks6t----', 'people who celebrate Earth Day all over the worlds6t----'])]
after cleaning:
[list(['students who study at Milford Central School, New York', 'people who want to attend Earth Day events in the Catskill region', 'adults who protect the environment in New York', 'people who celebrate Earth Day all over the world'])]


### write sft_input and sft_output

In [29]:
#all questions have 4 choices
test_df['n_choices'] = test_df.apply(lambda row: len(row['options']), axis=1).tolist()
print(set(test_df['n_choices']))

{4}


In [30]:
test_df['sft_input'] = test_df.apply(write_sft_input, axis=1)
test_df['sft_output'] = test_df.apply(write_sft_output, axis=1)

### check length of sft_input only -- since this is test set

In [31]:
print(test_df['sft_input'].str.len().describe()) #all under max length
print(max_length)


count    4934.000000
mean     1828.592420
std       597.240244
min       348.000000
25%      1472.000000
50%      1853.500000
75%      2147.000000
max      4900.000000
Name: sft_input, dtype: float64
6144


### save as json for cot eval

In [32]:
#add question id
test_df['id_new'] = list(range(1, test_df.shape[0]+1))
test_df['id_new'] = test_df['id_new'].astype(str) + '-' + test_df['id'].str[:-4]


In [33]:
#look at example
idx = 745
print('sft_input:')
print(test_df.iloc[idx]['sft_input'])

print('\n\nsft_output:')
print(test_df.iloc[idx]['sft_output'])

sft_input:
Answer the question below based only on the provided passage.

Passage: Many of you may wonder what else to do besides watching TV or surfing the Internet on weekends.Why not have a picnic? Junior 2 students at Beijing No.4 Middle Schoo1 had a "King of cooking" competition*
Earlier this month,about 300 students at the schoo1 went to a suburb of Beijing to have the contest.They were divided into 24  groups.Each group had buyers,slicers   ,firemakers.washers and cooks.
Firemakers faced the most problems during the time.Some of them had no idea how to keep fire burning. "The fire kept going out.we had to blow at the sparks and put on corn leaves and old newspapers,"said Wu Mofei,13.
"It took us an hour to make the fire.Our eyes had tears from all the smoke and our faces became dirty,"he added.
When the fires were finally made,the cooks became the busiest people.Huang Lanye made fried celery and ham pickled cabbage and tomato soup.
She was proud of her work."It's my first time m

In [34]:
### create jsonl file 
### MOVE BY HAND TO CORRECT FOLDER: learn-better/evolm/evaluation/cot-eval-harness/data/cooked

#format data for sft
data = []

# Write JSONL file
with open("data/race.jsonl", "w", encoding="utf-8") as f:
    for idx in tqdm(range(test_df.shape[0])):
        row = test_df.iloc[idx]
        item = {
            "id": str(row["id_new"]),
            "problem": str(row["sft_input"]),
            "gt_solution": str(row["sft_output"]), #not used because gt_answer is not null for evaluate_responses.py
            "gt_answer": str(row["answer"])
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Final # of samples in json file:", len(data))
print("Complete!")

100%|██████████████████████████████████████████████████████████| 4934/4934 [00:00<00:00, 15876.25it/s]

Final # of samples in json file: 0
Complete!
